# Generation Quality Evaluation Notebook
## Clinical Trials RAG — MSc Generative AI, University of Warwick

### Scoring Rubric (LLM-assisted evaluation)

| Metric | 0 | 0.5 | 1 |
|--------|---|-----|---|
| **factual_accuracy** | Wrong or hallucinated | — | Correct |
| **grounding** | Vague or uses outside knowledge | — | Supported by context, cites NCT IDs |
| **hallucination** | Invented details detected | — | Clean (no fabrication) |
| **completeness** | Fails completely | Partial answer | Fully answers query |

> **Note**: hallucination uses inverted scale: 0 = clean (good), 1 = hallucinated (bad)

In [1]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

print(f"Working directory: {Path.cwd()}")

Working directory: /Users/sohail/Documents/clinical-trials-rag


In [2]:
import json
from pathlib import Path
import glob

raw_files = sorted(Path("data/raw").glob("trials_full_*.json"))
trials_path = raw_files[-1]
with open(trials_path) as f:
    trials = json.load(f)
trial_lookup = {t["nct_id"]: t for t in trials}

ablation_files = sorted(Path("evaluation").glob("ablation_results_*.json"))
ablation_path = ablation_files[-1]
with open(ablation_path) as f:
    ablation_results = json.load(f)

print(f"Loaded {len(trials)} trials from {trials_path.name}")
print(f"Loaded {len(ablation_results)} ablation results from {ablation_path.name}")

Loaded 13967 trials from trials_full_20260330_232310.json
Loaded 100 ablation results from ablation_results_20260401_151004.json


In [3]:
def show_comparison(qid: str):
    """Show baseline vs full system answers side by side."""
    baseline = [r for r in ablation_results if r["query_id"] == qid and r["config"] == "1_baseline_llm"]
    full = [r for r in ablation_results if r["query_id"] == qid and r["config"] == "5_full_system"]
    if not baseline or not full:
        print(f"No results found for {qid}")
        return
    b, f_ = baseline[0], full[0]
    print("=" * 80)
    print(f"[{qid}] ({b.get('query_type', '?')}) {b['query']}")
    print("=" * 80)
    print("\n--- BASELINE (no retrieval) ---")
    print(b.get("answer", "N/A")[:500])
    print("\n--- FULL SYSTEM (Hybrid + Reranker + HyDE) ---")
    print(f_.get("answer", "N/A")[:500])
    print(f"\nRetrieved NCTs: {f_.get('retrieved_nct_ids', [])}")
    print(f"P@5: {f_.get('precision_at_5', 'N/A')}, R@5: {f_.get('recall_at_5', 'N/A')}")

def get_ground_truth(nct_id: str):
    t = trial_lookup.get(nct_id)
    if not t:
        print(f"{nct_id} not found in dataset")
        return
    print(f"\n[GT] {nct_id}: {t['brief_title']}")
    print(f"     Status: {t['overall_status']} | Phase: {t['phase']}")
    print(f"     Interventions: {t['intervention_names']}")
    print(f"     Conditions: {t['conditions']}")

def get_ground_truth_filter(**kwargs):
    matches = trials
    for k, v in kwargs.items():
        if k == "condition":
            matches = [t for t in matches if any(v.lower() in c.lower() for c in t.get("conditions", []))]
        elif k == "status":
            matches = [t for t in matches if t.get("overall_status") == v]
        elif k == "phase":
            matches = [t for t in matches if v in t.get("phase", "")]
    print(f"\n[Filter] {kwargs} => {len(matches)} matches")

In [4]:
eval_scores = []

In [5]:
# === Q01 (factual) ===
show_comparison("Q01")
get_ground_truth("NCT03970447")

# Score baseline
eval_scores.append({
    "query_id": "Q01",
    "config": "1_baseline_llm",
    "factual_accuracy": 0,
    "grounding": 0,
    "hallucination": 1,
    "completeness": 0,
    "notes": "Hallucinated: says NCT03970447 is a Tislelizumab esophageal cancer study. Actually GBM AGILE glioblastoma trial."
})

# Score full system
eval_scores.append({
    "query_id": "Q01",
    "config": "5_full_system",
    "factual_accuracy": 1,
    "grounding": 1,
    "hallucination": 0,
    "completeness": 1,
    "notes": "Correctly identifies GBM AGILE trial with accurate title, RECRUITING status, and intervention details."
})

[Q01] (factual) What is the official title and status of trial NCT03970447?

--- BASELINE (no retrieval) ---
The official title of trial NCT03970447 is "A Study to Evaluate the Safety and Efficacy of Tislelizumab in Combination With Chemotherapy in Patients With Advanced Esophageal Squamous Cell Carcinoma." The status of the trial is "Active, not recruiting."

--- FULL SYSTEM (Hybrid + Reranker + HyDE) ---
The official title of trial NCT03970447 is "GBM AGILE: Global Adaptive Trial Master Protocol: An International, Seamless Phase II/III Response Adaptive Randomization Platform Trial Designed To Evaluate Multiple Regimens In Newly Diagnosed and Recurrent GBM." The current status of the trial is "RECRUITING."

Retrieved NCTs: ['NCT03970447', 'NCT03970447', 'NCT03970447', 'NCT03970447', 'NCT03970447']
P@5: 0.2, R@5: 1.0

[GT] NCT03970447: A Trial to Evaluate Multiple Regimens in Newly Diagnosed and Recurrent Glioblastoma
     Status: RECRUITING | Phase: PHASE2, PHASE3
     Interventions:

In [6]:
# === Q02 (factual) ===
show_comparison("Q02")
get_ground_truth("NCT02685605")

# Score baseline
eval_scores.append({
    "query_id": "Q02",
    "config": "1_baseline_llm",
    "factual_accuracy": 0,
    "grounding": 0,
    "hallucination": 1,
    "completeness": 0,
    "notes": "Hallucinated: says Atezolizumab for solid tumors. Actually INTRAGO II glioblastoma intraoperative radiotherapy trial."
})

# Score full system
eval_scores.append({
    "query_id": "Q02",
    "config": "5_full_system",
    "factual_accuracy": 1,
    "grounding": 1,
    "hallucination": 0,
    "completeness": 1,
    "notes": "Correctly identifies INTRAGO II and lists all 4 interventions (surgery, IORT, radiochemotherapy, temozolomide)."
})

[Q02] (factual) What interventions are used in trial NCT02685605?

--- BASELINE (no retrieval) ---
The trial NCT02685605 involves the use of the intervention "Atezolizumab" (also known as MPDL3280A), which is an anti-PD-L1 monoclonal antibody. This intervention is being studied in combination with other treatments for patients with advanced solid tumors. If you need more specific details about the trial or its design, please let me know!

--- FULL SYSTEM (Hybrid + Reranker + HyDE) ---
In trial NCT02685605, titled "A Multicenter Randomized Phase III Trial on INTraoperative RAdiotherapy in Newly Diagnosed GliOblastoma Multiforme (INTRAGO II)," the following interventions are used:

1. **Standard surgery (PROCEDURE)** - This is the initial surgical intervention performed on patients.
2. **Intraoperative radiotherapy (RADIATION)** - This intervention involves administering radiation therapy during the surgical procedure.
3. **Radiochemotherapy (RADIATION)** - This combines radiat

Retrieve

In [7]:
# === Q03 (filtering) ===
show_comparison("Q03")
get_ground_truth_filter(condition="glioblastoma", status="RECRUITING", phase="PHASE3")

# Score baseline
eval_scores.append({
    "query_id": "Q03",
    "config": "1_baseline_llm",
    "factual_accuracy": 0,
    "grounding": 0,
    "hallucination": 0,
    "completeness": 0,
    "notes": "Generic refusal; no actual trials listed."
})

# Score full system
eval_scores.append({
    "query_id": "Q03",
    "config": "5_full_system",
    "factual_accuracy": 1,
    "grounding": 1,
    "hallucination": 0,
    "completeness": 0.5,
    "notes": "Lists 5 phase 3 recruiting GBM trials with accurate details. Partial: 5 of 14 total matches."
})

[Q03] (filtering) What phase 3 glioblastoma trials are currently recruiting?

--- BASELINE (no retrieval) ---
I don't have real-time access to current clinical trial databases or the internet to provide the latest information on ongoing phase 3 glioblastoma trials. However, you can find up-to-date information on clinical trials by visiting websites such as:

1. **ClinicalTrials.gov** - A database of privately and publicly funded clinical studies conducted around the world.
2. **World Health Organization International Clinical Trials Registry Platform (WHO ICTRP)** - A platform that aims to ensure that a

--- FULL SYSTEM (Hybrid + Reranker + HyDE) ---
Here are the currently recruiting Phase 3 clinical trials for glioblastoma based on the provided context:

1. **NCT ID: NCT06556563**
   - **Title:** EF-41/KEYNOTE D58: Phase 3 Study of Optune Concomitant With Temozolomide Plus Pembrolizumab in Newly Diagnosed Glioblastoma
   - **Official Title:** A Phase 3, Randomized, Double-Blind, Place

In [8]:
# === Q04 (filtering) ===
show_comparison("Q04")
get_ground_truth_filter(condition="ovarian", status="COMPLETED", phase="PHASE2")

# Score baseline
eval_scores.append({
    "query_id": "Q04",
    "config": "1_baseline_llm",
    "factual_accuracy": 0,
    "grounding": 0,
    "hallucination": 0,
    "completeness": 0,
    "notes": "Generic search instructions; no specific trials."
})

# Score full system
eval_scores.append({
    "query_id": "Q04",
    "config": "5_full_system",
    "factual_accuracy": 1,
    "grounding": 1,
    "hallucination": 0,
    "completeness": 0.5,
    "notes": "Lists 3 completed phase 2 ovarian trials with details. Partial: small subset of 724 matches."
})

[Q04] (filtering) Show me completed phase 2 ovarian cancer trials

--- BASELINE (no retrieval) ---
To find completed Phase 2 clinical trials for ovarian cancer, you can search clinical trial registries such as ClinicalTrials.gov or the European Union Clinical Trials Register. These databases provide detailed information about trials, including their status, results, and specific conditions being studied.

As of my last update in October 2023, I cannot access real-time databases or provide specific trial listings. However, you can follow these steps to find the information you need:

1. **Visi

--- FULL SYSTEM (Hybrid + Reranker + HyDE) ---
Here are the completed Phase 2 clinical trials for ovarian cancer based on the provided context:

### NCT ID: NCT01616303
- **Title:** A Controlled Study of the Effectiveness of Oregovomab (Antibody) Plus Chemotherapy in Advanced Ovarian Cancer
- **Official Title:** Phase 2: A Randomized Controlled Study on Effectiveness of Chemotherapy (Carboplatin-

In [9]:
# === Q05 (filtering) ===
show_comparison("Q05")
get_ground_truth_filter(condition="pancreatic", status="RECRUITING")

# Score baseline
eval_scores.append({
    "query_id": "Q05",
    "config": "1_baseline_llm",
    "factual_accuracy": 0,
    "grounding": 0,
    "hallucination": 0,
    "completeness": 0,
    "notes": "Generic refusal; no trials listed."
})

# Score full system
eval_scores.append({
    "query_id": "Q05",
    "config": "5_full_system",
    "factual_accuracy": 1,
    "grounding": 1,
    "hallucination": 0,
    "completeness": 0.5,
    "notes": "Lists 5 recruiting pancreatic trials with concrete details. Partial: small subset of 904 matches."
})

[Q05] (filtering) List all recruiting pancreatic cancer trials

--- BASELINE (no retrieval) ---
I don't have real-time access to databases or the internet to provide a current list of recruiting pancreatic cancer trials. However, you can find this information on clinical trial registries such as:

1. **ClinicalTrials.gov** - A database of privately and publicly funded clinical studies conducted around the world. You can search for pancreatic cancer trials and filter by recruitment status.

2. **World Health Organization International Clinical Trials Registry Platform (ICTRP)** - This platf

--- FULL SYSTEM (Hybrid + Reranker + HyDE) ---
Here are the recruiting clinical trials for pancreatic cancer based on the provided context:

### NCT ID: NCT05685602
- **Title:** CA-4948 Added to Standard Chemotherapy to Treat Metastatic or Unresectable Pancreatic Cancer
- **Official Title:** A Phase I Clinical Trial of CA-4948 in Combination With Gemcitabine and Nab-Paclitaxel in Metastatic or Unres

In [10]:
# === Q06 (filtering) ===
show_comparison("Q06")
get_ground_truth_filter(condition="small cell lung", status="NOT_YET_RECRUITING")

# Score baseline
eval_scores.append({
    "query_id": "Q06",
    "config": "1_baseline_llm",
    "factual_accuracy": 0,
    "grounding": 0,
    "hallucination": 0,
    "completeness": 0,
    "notes": "Generic refusal; no specific trial data."
})

# Score full system
eval_scores.append({
    "query_id": "Q06",
    "config": "5_full_system",
    "factual_accuracy": 1,
    "grounding": 1,
    "hallucination": 0,
    "completeness": 0.5,
    "notes": "Lists 2 not-yet-recruiting SCLC trials with NCT IDs and details. Partial: subset of 525 matches."
})

[Q06] (filtering) What SCLC trials are not yet recruiting?

--- BASELINE (no retrieval) ---
To find specific details about small cell lung cancer (SCLC) trials that are not yet recruiting, you would typically need to consult clinical trial registries such as ClinicalTrials.gov or other databases that track ongoing and upcoming clinical trials. These platforms allow you to filter trials based on their recruitment status, location, phase, and other criteria.

As of my last update in October 2023, I do not have real-time access to databases or the ability to browse the internet for the la

--- FULL SYSTEM (Hybrid + Reranker + HyDE) ---
The following small cell lung cancer (SCLC) trials are currently listed as not yet recruiting:

1. **NCT ID: NCT07103408**
   - **Title:** A Study Evaluating Concurrent Chemoradiotherapy Combined With Dual Immune Checkpoint Blockade for Limited-stage Small Cell Lung Cancer
   - **Phase:** Phase II
   - **Status:** Not yet recruiting
   - **Interventions:** 

In [11]:
# === Q07 (semantic) ===
show_comparison("Q07")

# Score baseline
eval_scores.append({
    "query_id": "Q07",
    "config": "1_baseline_llm",
    "factual_accuracy": 0,
    "grounding": 0,
    "hallucination": 1,
    "completeness": 0.5,
    "notes": "Lists generic immunotherapy categories but no specific trials. Unsupported claims."
})

# Score full system
eval_scores.append({
    "query_id": "Q07",
    "config": "5_full_system",
    "factual_accuracy": 1,
    "grounding": 1,
    "hallucination": 0,
    "completeness": 0.5,
    "notes": "Found 4 pembrolizumab trials for recurrent GBM (implies prior TMZ). Partial: TMZ failure criterion not explicitly confirmed in trial descriptions."
})

[Q07] (semantic) What immunotherapy trials are available for glioblastoma patients who failed temozolomide?

--- BASELINE (no retrieval) ---
As of my last update in October 2023, there are several immunotherapy trials available for glioblastoma patients who have failed temozolomide. These trials may include various approaches such as checkpoint inhibitors, vaccine therapies, oncolytic virus therapies, and adoptive cell therapies. Here are some general categories of immunotherapy trials that may be available:

1. **Checkpoint Inhibitors**: Trials investigating drugs that block immune checkpoints (e.g., PD-1/PD-L1 inhibitors like nivol

--- FULL SYSTEM (Hybrid + Reranker + HyDE) ---
Based on the provided clinical trial context, here are the immunotherapy trials available for glioblastoma patients who have failed temozolomide:

### NCT04977375
- **Official Title**: Phase Ib/II Trial of Anti-PD-1 Immunotherapy and Stereotactic Radiation in Patients With Recurrent Glioblastoma
- **Phase**: 

In [12]:
# === Q08 (semantic) ===
show_comparison("Q08")

# Score baseline
eval_scores.append({
    "query_id": "Q08",
    "config": "1_baseline_llm",
    "factual_accuracy": 1,
    "grounding": 0,
    "hallucination": 0,
    "completeness": 0.5,
    "notes": "Correct general yes-answer but generic; no specific trials cited."
})

# Score full system
eval_scores.append({
    "query_id": "Q08",
    "config": "5_full_system",
    "factual_accuracy": 1,
    "grounding": 1,
    "hallucination": 0,
    "completeness": 1,
    "notes": "Correctly identifies 4 specific CAR-T pancreatic cancer trials with NCT IDs and detailed descriptions."
})

[Q08] (semantic) Are there any CAR-T cell therapy trials for pancreatic cancer?

--- BASELINE (no retrieval) ---
Yes, there are ongoing clinical trials investigating CAR-T cell therapy for pancreatic cancer. While CAR-T cell therapy has shown significant success in certain hematological malignancies, its application in solid tumors like pancreatic cancer is still being explored. Researchers are working on developing CAR-T cells that can target specific antigens expressed on pancreatic cancer cells.

To find specific trials, you can search clinical trial registries such as ClinicalTrials.gov, where you can 

--- FULL SYSTEM (Hybrid + Reranker + HyDE) ---
Yes, there are several CAR-T cell therapy trials for pancreatic cancer based on the provided context. Here are the details:

### 1. NCT ID: NCT07066995
- **Title**: Dual-Target Chimeric Antigen Receptor (CAR) T-Cell Therapy Directed Against Mesothelin and Claudin 18.2 in Patients With Advanced or Metastatic Pancreatic Cancer
- **Phase**

In [13]:
# === Q09 (semantic) ===
show_comparison("Q09")

# Score baseline
eval_scores.append({
    "query_id": "Q09",
    "config": "1_baseline_llm",
    "factual_accuracy": 0,
    "grounding": 0,
    "hallucination": 1,
    "completeness": 0.5,
    "notes": "Lists generic PARP inhibitor drug names without any trial IDs or grounding."
})

# Score full system
eval_scores.append({
    "query_id": "Q09",
    "config": "5_full_system",
    "factual_accuracy": 1,
    "grounding": 1,
    "hallucination": 0,
    "completeness": 1,
    "notes": "Lists 3 specific olaparib ovarian cancer trials with NCT IDs and trial details from retrieved context."
})

[Q09] (semantic) Which trials are investigating PARP inhibitors for ovarian cancer?

--- BASELINE (no retrieval) ---
As of my last update in October 2023, several clinical trials are investigating PARP inhibitors for ovarian cancer. These trials typically focus on various aspects, including the efficacy of PARP inhibitors as monotherapy, their use in combination with other treatments, and their effectiveness in different patient populations, such as those with BRCA mutations or homologous recombination deficiency (HRD).

Some notable PARP inhibitors that have been studied include:

1. **Olaparib (Lynparza)** -

--- FULL SYSTEM (Hybrid + Reranker + HyDE) ---
The following clinical trials are investigating PARP inhibitors for ovarian cancer:

### NCT ID: NCT02681237
- **Title**: A Study of Cediranib and Olaparib at Disease Worsening in Ovarian Cancer
- **Phase**: NA
- **Status**: COMPLETED
- **Interventions**: Combination of Cediranib and Olaparib
- **Conditions**: Ovarian Cancer
- **Sta

In [14]:
# === Q10 (semantic) ===
show_comparison("Q10")

# Score baseline
eval_scores.append({
    "query_id": "Q10",
    "config": "1_baseline_llm",
    "factual_accuracy": 1,
    "grounding": 0,
    "hallucination": 0,
    "completeness": 0.5,
    "notes": "Lists plausible checkpoint inhibitor combos for SCLC (nivo+ipi, atezo+chemo) but no trial-specific grounding."
})

# Score full system
eval_scores.append({
    "query_id": "Q10",
    "config": "5_full_system",
    "factual_accuracy": 1,
    "grounding": 1,
    "hallucination": 0,
    "completeness": 0.5,
    "notes": "Found 2 checkpoint inhibitor trials: BIOLUMA (nivo+ipi) and CheckMate 451 (nivo+/-ipi). Partial: only 2 combos, both nivolumab-based."
})

[Q10] (semantic) What checkpoint inhibitor combinations are being tested in small cell lung cancer?

--- BASELINE (no retrieval) ---
As of my last update in October 2023, several checkpoint inhibitor combinations are being tested in small cell lung cancer (SCLC). These combinations typically involve immune checkpoint inhibitors, such as anti-PD-1/PD-L1 antibodies and anti-CTLA-4 antibodies, often in conjunction with chemotherapy or other targeted therapies. Some notable combinations include:

1. **Nivolumab (anti-PD-1) + Ipilimumab (anti-CTLA-4)**: This combination has been explored in various clinical trials to evaluate its 

--- FULL SYSTEM (Hybrid + Reranker + HyDE) ---
Two clinical trials are testing checkpoint inhibitor combinations in small cell lung cancer (SCLC):

1. **NCT03083691**
   - **Title**: BIOLUMA: Biomarkers for Nivolumab and Ipilimumab and Evaluation of the Combination in Lung Cancer
   - **Phase**: Phase II
   - **Status**: Completed
   - **Interventions**: 
     - 

In [15]:
# === Q11 (comparative) ===
show_comparison("Q11")

# Score baseline
eval_scores.append({
    "query_id": "Q11",
    "config": "1_baseline_llm",
    "factual_accuracy": 1,
    "grounding": 0,
    "hallucination": 1,
    "completeness": 0.5,
    "notes": "Generic comparison but unsupported; ovarian adoptive cell therapy claim is invented without evidence."
})

# Score full system
eval_scores.append({
    "query_id": "Q11",
    "config": "5_full_system",
    "factual_accuracy": 0,
    "grounding": 0,
    "hallucination": 0,
    "completeness": 0,
    "notes": "FAILURE: Says Insufficient information. Retrieved trials include immunotherapy from both conditions but the model failed to synthesise a comparison."
})

[Q11] (comparative) Compare the types of immunotherapy approaches being used in glioblastoma versus ovarian cancer trials

--- BASELINE (no retrieval) ---
Immunotherapy approaches for glioblastoma and ovarian cancer differ significantly due to the distinct tumor microenvironments, immune evasion mechanisms, and biological characteristics of these cancers. Here’s a comparison of the types of immunotherapy being explored in clinical trials for both types of cancer:

### Glioblastoma Immunotherapy Approaches

1. **Checkpoint Inhibitors**: 
   - **PD-1/PD-L1 Inhibitors**: Drugs like pembrolizumab and nivolumab are being tested, often in combination wi

--- FULL SYSTEM (Hybrid + Reranker + HyDE) ---
Insufficient information in the current knowledge base.

Retrieved NCTs: ['NCT04396860', 'NCT04145115', 'NCT02852655', 'NCT06097975', 'NCT04817254']
P@5: -1.0, R@5: -1.0


In [16]:
# === Q12 (comparative) ===
show_comparison("Q12")
get_ground_truth_filter(condition="pancreatic", phase="PHASE3")
get_ground_truth_filter(condition="small cell lung", phase="PHASE3")

# Score baseline
eval_scores.append({
    "query_id": "Q12",
    "config": "1_baseline_llm",
    "factual_accuracy": 1,
    "grounding": 0,
    "hallucination": 0,
    "completeness": 0.5,
    "notes": "Correctly states SCLC generally has more phase 3 trials. Generic with no specific numbers."
})

# Score full system
eval_scores.append({
    "query_id": "Q12",
    "config": "5_full_system",
    "factual_accuracy": 0,
    "grounding": 1,
    "hallucination": 1,
    "completeness": 0.5,
    "notes": "WRONG: Claims pancreatic has 4 phase 3 trials and no SCLC found. Dataset shows SCLC=808 vs pancreatic=197. Retrieval only returned pancreatic trials."
})

[Q12] (comparative) Which cancer type has more phase 3 trials — pancreatic cancer or SCLC?

--- BASELINE (no retrieval) ---
As of my last update in October 2023, small cell lung cancer (SCLC) generally has more phase 3 trials compared to pancreatic cancer. SCLC is a more common type of lung cancer and has been the focus of numerous clinical trials due to its aggressive nature and the need for effective treatments. Pancreatic cancer, while also a significant area of research, typically has fewer phase 3 trials in comparison. However, the exact number of trials can vary over time, so it's always a good idea to check a 

--- FULL SYSTEM (Hybrid + Reranker + HyDE) ---
Based on the provided clinical trial context, there are a total of **four Phase 3 trials** related to pancreatic cancer and **zero Phase 3 trials** related to small cell lung cancer (SCLC). Here are the details of the Phase 3 trials for pancreatic cancer:

1. **NCT ID: NCT00058201**
   - **Title:** Two Chemotherapy Regimens C

In [17]:
# === Q13 (temporal) ===
show_comparison("Q13")
get_ground_truth_filter(condition="glioblastoma")

# Score baseline
eval_scores.append({
    "query_id": "Q13",
    "config": "1_baseline_llm",
    "factual_accuracy": 0,
    "grounding": 0,
    "hallucination": 0,
    "completeness": 0,
    "notes": "Generic refusal; says it cannot access real-time data."
})

# Score full system
eval_scores.append({
    "query_id": "Q13",
    "config": "5_full_system",
    "factual_accuracy": 1,
    "grounding": 1,
    "hallucination": 0,
    "completeness": 0.5,
    "notes": "Found 2 GBM trials with verified 2024 start dates (NCT06258018 Sept 2024, NCT06388733 June 2024). Partial: subset of 261 matches."
})

[Q13] (temporal) What glioblastoma trials started in 2024 or later?

--- BASELINE (no retrieval) ---
I'm sorry, but I don't have access to real-time data or updates beyond October 2023, so I cannot provide information on glioblastoma trials that started in 2024 or later. For the most current information on clinical trials, I recommend checking clinical trial registries such as ClinicalTrials.gov or consulting recent publications in medical journals.

--- FULL SYSTEM (Hybrid + Reranker + HyDE) ---
The following glioblastoma trials started in 2024 or later:

1. **NCT ID: NCT06258018**
   - **Title:** Niraparib and Temozolomide in Patients Glioblastoma
   - **Phase:** PHASE1, PHASE2
   - **Status:** NOT_YET_RECRUITING
   - **Interventions:** Niraparib plus Temozolomide "One Week on, One Week Off"
   - **Conditions:** Recurrent Isocitrate Dehydrogenase (IDH) Wild Type Glioblastoma and IDH Mutant Gliomas
   - **Start Date:** 2024-09
   - **Primary Completion Date:** 2026-12
   - **Last Upda

In [18]:
# === Q14 (temporal) ===
show_comparison("Q14")
get_ground_truth("NCT05799274")
get_ground_truth("NCT05554367")

# Score baseline
eval_scores.append({
    "query_id": "Q14",
    "config": "1_baseline_llm",
    "factual_accuracy": 0,
    "grounding": 0,
    "hallucination": 0,
    "completeness": 0,
    "notes": "Generic refusal; recommends visiting ClinicalTrials.gov."
})

# Score full system
eval_scores.append({
    "query_id": "Q14",
    "config": "5_full_system",
    "factual_accuracy": 0,
    "grounding": 1,
    "hallucination": 1,
    "completeness": 0.5,
    "notes": "Lists 5 ovarian trials grounded in retrieved data, but most recently updated ordering is unsupported. Only 2 trials show last_update dates; ranking claim is fabricated."
})

[Q14] (temporal) What are the 5 most recently updated ovarian cancer trials?

--- BASELINE (no retrieval) ---
I don't have real-time access to databases or the internet to provide the most current updates on clinical trials. However, you can find the most recently updated ovarian cancer trials by visiting clinical trial registries such as:

1. **ClinicalTrials.gov** - A database of privately and publicly funded clinical studies conducted around the world.
2. **European Union Clinical Trials Register** - Provides information on interventional clinical trials conducted in the European Union member states 

--- FULL SYSTEM (Hybrid + Reranker + HyDE) ---
Here are the 5 most recently updated ovarian cancer trials based on the provided context:

1. **NCT ID: NCT05605535**
   - **Title:** A Phase 2, Double-Blind, Placebo-Controlled, Multicenter Clinical Trial Comparing Chemoimmunotherapy (Paclitaxel-Carboplatin-Oregovomab) Versus Chemotherapy (Paclitaxel-Carboplatin-Placebo) as Neoadjuvant Th

In [19]:
# === Q15 (semantic) ===
show_comparison("Q15")

# Score baseline
eval_scores.append({
    "query_id": "Q15",
    "config": "1_baseline_llm",
    "factual_accuracy": 1,
    "grounding": 0,
    "hallucination": 0,
    "completeness": 0.5,
    "notes": "Correctly says yes but generic; no specific trials cited."
})

# Score full system
eval_scores.append({
    "query_id": "Q15",
    "config": "5_full_system",
    "factual_accuracy": 1,
    "grounding": 1,
    "hallucination": 0,
    "completeness": 1,
    "notes": "Cites NCT03197506 (pembrolizumab+TMZ+radiation) and NCT04977375 (pembrolizumab+stereotactic radiation) for GBM. Fully answers the query."
})

[Q15] (semantic) Are there any trials combining radiation therapy with immunotherapy for glioblastoma?

--- BASELINE (no retrieval) ---
Yes, there are clinical trials that are investigating the combination of radiation therapy with immunotherapy for glioblastoma. This approach aims to enhance the effectiveness of treatment by using radiation to potentially increase the immune response against tumor cells. Various immunotherapeutic agents, such as checkpoint inhibitors, vaccines, and oncolytic viruses, are being studied in conjunction with radiation therapy.

To find specific ongoing trials, you can search clinical trial registri

--- FULL SYSTEM (Hybrid + Reranker + HyDE) ---
Yes, there are trials combining radiation therapy with immunotherapy for glioblastoma. Here are the relevant trials:

1. **NCT ID: NCT03197506**
   - **Phase:** II
   - **Status:** Not specified in the provided context
   - **Interventions:** Pembrolizumab (MK-3475) in combination with standard therapy (temozolomi

In [20]:
# === Q16 (semantic) ===
show_comparison("Q16")

# Score baseline
eval_scores.append({
    "query_id": "Q16",
    "config": "1_baseline_llm",
    "factual_accuracy": 1,
    "grounding": 0,
    "hallucination": 0,
    "completeness": 0.5,
    "notes": "Generic eligibility criteria categories; no trial-specific info."
})

# Score full system
eval_scores.append({
    "query_id": "Q16",
    "config": "5_full_system",
    "factual_accuracy": 1,
    "grounding": 1,
    "hallucination": 0,
    "completeness": 1,
    "notes": "Provides detailed eligibility criteria from 4 specific gemcitabine pancreatic cancer trials with NCT IDs."
})

[Q16] (semantic) What are the eligibility criteria for pancreatic cancer trials involving gemcitabine?

--- BASELINE (no retrieval) ---
Eligibility criteria for pancreatic cancer trials involving gemcitabine can vary depending on the specific study, but common criteria often include:

1. **Diagnosis**: Confirmed diagnosis of pancreatic adenocarcinoma or other specific types of pancreatic cancer.

2. **Stage of Cancer**: Typically, trials may focus on patients with locally advanced or metastatic disease, but some may include those with resectable tumors.

3. **Previous Treatments**: Criteria may specify whether patients can have 

--- FULL SYSTEM (Hybrid + Reranker + HyDE) ---
Here are the eligibility criteria for pancreatic cancer trials involving gemcitabine based on the provided context:

### NCT ID: NCT00471146
**Study Title:** Study Of Gemcitabine Plus AG-013736 Versus Gemcitabine For Advanced Pancreatic Cancer  
**Eligibility Criteria:**
- **Inclusion Criteria:**
  - Histologicall

In [21]:
# === Q17 (edge_case) ===
show_comparison("Q17")

# Score baseline
eval_scores.append({
    "query_id": "Q17",
    "config": "1_baseline_llm",
    "factual_accuracy": 1,
    "grounding": 0,
    "hallucination": 0,
    "completeness": 0.5,
    "notes": "Broad generic overview of brain tumor trials; no specific trials cited."
})

# Score full system
eval_scores.append({
    "query_id": "Q17",
    "config": "5_full_system",
    "factual_accuracy": 1,
    "grounding": 1,
    "hallucination": 0,
    "completeness": 0.5,
    "notes": "Successfully maps brain tumors to GBM via semantic search. Lists 5 glioblastoma trials. Partial: narrow subset of brain tumors."
})

[Q17] (edge_case) Tell me about clinical trials for brain tumors

--- BASELINE (no retrieval) ---
Clinical trials for brain tumors are research studies that evaluate new treatments, diagnostic methods, or interventions for various types of brain tumors, including gliomas, meningiomas, and metastatic brain tumors. These trials are essential for advancing medical knowledge and improving patient outcomes. Here are some key points about clinical trials for brain tumors:

1. **Types of Trials**: Clinical trials can be categorized into several phases:
   - **Phase I**: Focus on safety and dosage. 

--- FULL SYSTEM (Hybrid + Reranker + HyDE) ---
Here are the clinical trials for brain tumors based on the provided context:

### 1. NCT00093964
- **Title**: Cilengitide (EMD 121974) for Recurrent Glioblastoma Multiforme (Brain Tumor)
- **Phase**: Phase IIa
- **Status**: Completed
- **Intervention**: Cilengitide (EMD 121,974) administered as a single agent.
- **Conditions**: Recurrent Glioblastoma 

In [22]:
# === Q18 (edge_case) ===
show_comparison("Q18")

# Score baseline
eval_scores.append({
    "query_id": "Q18",
    "config": "1_baseline_llm",
    "factual_accuracy": 1,
    "grounding": 0,
    "hallucination": 0,
    "completeness": 0.5,
    "notes": "Generic overview of lung cancer trial types; no specific trials."
})

# Score full system
eval_scores.append({
    "query_id": "Q18",
    "config": "5_full_system",
    "factual_accuracy": 1,
    "grounding": 1,
    "hallucination": 0,
    "completeness": 0.5,
    "notes": "Found 5 specific lung cancer trials (NSCLC + SCLC mix) with NCT IDs. Partial: small subset of available trials."
})

[Q18] (edge_case) What trials are available for lung cancer?

--- BASELINE (no retrieval) ---
There are numerous clinical trials available for lung cancer, which can vary based on factors such as the type of lung cancer (e.g., non-small cell lung cancer, small cell lung cancer), stage of the disease, previous treatments, and specific patient characteristics. Here are some common types of trials you might find:

1. **Treatment Trials**: These trials test new treatments, including chemotherapy, targeted therapy, immunotherapy, or combinations of these. They may also evaluate new drug combi

--- FULL SYSTEM (Hybrid + Reranker + HyDE) ---
Here are the clinical trials available for lung cancer based on the provided context:

### NCT02733159
- **Title**: A Phase II Trial of Pembrolizumab in Patients With Non-small Cell Lung Cancer and a Performance Status of 2
- **Phase**: Phase II
- **Status**: Not specified
- **Intervention**: Pembrolizumab
- **Eligibility**: Patients with Non-Small Cell L

In [23]:
# === Q19 (edge_case) ===
show_comparison("Q19")

# Score baseline
eval_scores.append({
    "query_id": "Q19",
    "config": "1_baseline_llm",
    "factual_accuracy": 1,
    "grounding": 0,
    "hallucination": 0,
    "completeness": 0.5,
    "notes": "Generic yes-answer. Recommends trial registries."
})

# Score full system
eval_scores.append({
    "query_id": "Q19",
    "config": "5_full_system",
    "factual_accuracy": 1,
    "grounding": 1,
    "hallucination": 0,
    "completeness": 0.5,
    "notes": "Found 3 melanoma trials cross-listed in dataset (NCT03006887, NCT04205227, NCT03158935). Semantic search found relevant cross-condition trials."
})

[Q19] (edge_case) Are there any trials for melanoma?

--- BASELINE (no retrieval) ---
Yes, there are numerous clinical trials for melanoma. These trials may focus on various aspects of melanoma treatment, including new drug therapies, immunotherapies, combination therapies, and targeted therapies. They may also investigate different stages of melanoma, from early-stage to advanced metastatic melanoma.

To find specific trials, you can search clinical trial registries such as:

- **ClinicalTrials.gov**: A database of privately and publicly funded clinical studies conducted around 

--- FULL SYSTEM (Hybrid + Reranker + HyDE) ---
Yes, there are clinical trials for melanoma based on the provided context. Here are the details:

### NCT ID: NCT03006887
- **Phase**: Phase 1b
- **Status**: Not specified in the context
- **Interventions**: Lenvatinib plus Pembrolizumab
- **Eligibility**: Participants with selected solid tumors, including melanoma (excluding uveal melanoma).

### NCT ID: NCT0420

In [24]:
# === Q20 (edge_case) ===
show_comparison("Q20")

# Score baseline
eval_scores.append({
    "query_id": "Q20",
    "config": "1_baseline_llm",
    "factual_accuracy": 1,
    "grounding": 0,
    "hallucination": 0,
    "completeness": 1,
    "notes": "Correctly explains phase 5 does not exist; describes standard phase 1-4 framework."
})

# Score full system
eval_scores.append({
    "query_id": "Q20",
    "config": "5_full_system",
    "factual_accuracy": 0,
    "grounding": 0,
    "hallucination": 0,
    "completeness": 0,
    "notes": "Says Insufficient information without explaining that phase 5 does not exist. Baseline handles this better."
})

[Q20] (edge_case) What phase 5 trials exist for ovarian cancer?

--- BASELINE (no retrieval) ---
As of my last knowledge update in October 2023, there is no officially recognized "Phase 5" in clinical trials. Clinical trials are typically categorized into four phases:

- **Phase 1**: Focuses on safety and dosage.
- **Phase 2**: Examines efficacy and side effects.
- **Phase 3**: Compares the new treatment to standard treatments in larger populations.
- **Phase 4**: Conducted after a treatment has been approved to monitor long-term effects and effectiveness in the general population.

If you 

--- FULL SYSTEM (Hybrid + Reranker + HyDE) ---
Insufficient information in the current knowledge base.

Retrieved NCTs: ['NCT02082470', 'NCT05752448', 'NCT02082470', 'NCT02569983', 'NCT05752448']
P@5: -1.0, R@5: -1.0


In [25]:
import json
with open("evaluation/eval_scores.json", "w") as f:
    json.dump(eval_scores, f, indent=2)
print(f"Saved {len(eval_scores)} scores")

Saved 40 scores


In [26]:
scored_baseline = [s for s in eval_scores if s["config"] == "1_baseline_llm" and s["factual_accuracy"] is not None]
scored_full = [s for s in eval_scores if s["config"] == "5_full_system" and s["factual_accuracy"] is not None]

def avg(scores, key):
    vals = [s[key] for s in scores if s[key] is not None]
    return sum(vals) / len(vals) if vals else float("nan")

metrics = ["factual_accuracy", "grounding", "hallucination", "completeness"]

print("=" * 65)
print("MANUAL EVALUATION SUMMARY")
print("=" * 65)
print(f"{'Metric':<22} {'Baseline':>12} {'Full System':>12} {'Delta':>10}")
print("-" * 65)

for m in metrics:
    b_avg = avg(scored_baseline, m)
    f_avg = avg(scored_full, m)
    delta = f_avg - b_avg
    if m == "hallucination":
        direction = "\u2191" if delta < 0 else "\u2193" if delta > 0 else "="
    else:
        direction = "\u2191" if delta > 0 else "\u2193" if delta < 0 else "="
    print(f"{m:<22} {b_avg:>12.3f} {f_avg:>12.3f} {delta:>+9.3f} {direction}")

print("=" * 65)
print(f"Scored queries: {len(scored_baseline)} baseline, {len(scored_full)} full system")
print()
print("Note: hallucination uses inverted scale (lower = better)")
print("Delta direction: \u2191 = improvement for that metric")

MANUAL EVALUATION SUMMARY
Metric                     Baseline  Full System      Delta
-----------------------------------------------------------------
factual_accuracy              0.500        0.800    +0.300 ↑
grounding                     0.000        0.900    +0.900 ↑
hallucination                 0.250        0.100    -0.150 ↑
completeness                  0.325        0.600    +0.275 ↑
Scored queries: 20 baseline, 20 full system

Note: hallucination uses inverted scale (lower = better)
Delta direction: ↑ = improvement for that metric


In [27]:
from collections import defaultdict

type_scores = defaultdict(lambda: {"baseline": [], "full": []})
for s in eval_scores:
    qtype = [r for r in ablation_results if r["query_id"] == s["query_id"]][0].get("query_type", "unknown")
    config_key = "baseline" if s["config"] == "1_baseline_llm" else "full"
    type_scores[qtype][config_key].append(s)

print("\n" + "=" * 75)
print("PER-QUERY-TYPE BREAKDOWN")
print("=" * 75)

for qtype in ["factual", "filtering", "semantic", "comparative", "temporal", "edge_case"]:
    if qtype not in type_scores:
        continue
    bl_scores = type_scores[qtype]["baseline"]
    fs_scores = type_scores[qtype]["full"]
    bl_fa = sum(s["factual_accuracy"] for s in bl_scores) / len(bl_scores)
    fs_fa = sum(s["factual_accuracy"] for s in fs_scores) / len(fs_scores)
    bl_ha = sum(s["hallucination"] for s in bl_scores) / len(bl_scores)
    fs_ha = sum(s["hallucination"] for s in fs_scores) / len(fs_scores)
    bl_co = sum(s["completeness"] for s in bl_scores) / len(bl_scores)
    fs_co = sum(s["completeness"] for s in fs_scores) / len(fs_scores)
    n = len(bl_scores)
    print(f"\n{qtype.upper()} ({n} queries):")
    print(f"  Factual Accuracy:  BL={bl_fa:.2f}  FS={fs_fa:.2f}  delta={fs_fa-bl_fa:+.2f}")
    print(f"  Hallucination:     BL={bl_ha:.2f}  FS={fs_ha:.2f}  delta={fs_ha-bl_ha:+.2f}")
    print(f"  Completeness:      BL={bl_co:.2f}  FS={fs_co:.2f}  delta={fs_co-bl_co:+.2f}")

print("\n" + "=" * 75)


PER-QUERY-TYPE BREAKDOWN

FACTUAL (2 queries):
  Factual Accuracy:  BL=0.00  FS=1.00  delta=+1.00
  Hallucination:     BL=1.00  FS=0.00  delta=-1.00
  Completeness:      BL=0.00  FS=1.00  delta=+1.00

FILTERING (4 queries):
  Factual Accuracy:  BL=0.00  FS=1.00  delta=+1.00
  Hallucination:     BL=0.00  FS=0.00  delta=+0.00
  Completeness:      BL=0.00  FS=0.50  delta=+0.50

SEMANTIC (6 queries):
  Factual Accuracy:  BL=0.67  FS=1.00  delta=+0.33
  Hallucination:     BL=0.33  FS=0.00  delta=-0.33
  Completeness:      BL=0.50  FS=0.83  delta=+0.33

COMPARATIVE (2 queries):
  Factual Accuracy:  BL=1.00  FS=0.00  delta=-1.00
  Hallucination:     BL=0.50  FS=0.50  delta=+0.00
  Completeness:      BL=0.50  FS=0.25  delta=-0.25

TEMPORAL (2 queries):
  Factual Accuracy:  BL=0.00  FS=0.50  delta=+0.50
  Hallucination:     BL=0.00  FS=0.50  delta=+0.50
  Completeness:      BL=0.00  FS=0.50  delta=+0.50

EDGE_CASE (4 queries):
  Factual Accuracy:  BL=1.00  FS=0.75  delta=-0.25
  Hallucination: